# Used Car EDA

Exploratory Data Analysis of used-car listings using **Pandas, Matplotlib, and Seaborn**.

**Goal:** Explore how listed prices vary with brand, fuel type, manufacturing year, mileage, and city.

**Dataset:** [Used Car Information on Kaggle](https://www.kaggle.com/datasets/sanjeetsinghnaik/used-car-information). Download Dataset.csv before running the notebook. The raw CSV is not stored in this repository.

## 1. Import libraries and load the data

The code checks a few common locations so the notebook can run from the learning folder or the project folder.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

possible_paths = [
    Path("data/Dataset.csv"),
    Path("Datasets for DS/Dataset.csv"),
    Path("Dataset.csv"),
    Path("../Datasets for DS/Dataset.csv"),
]
data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset.csv was not found. Download it from Kaggle and place it "
        "in the data folder beside this notebook, or update possible_paths."
    )

df = pd.read_csv(data_path)
print("Loaded:", data_path)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. First data-quality checks

First check missing values and exact repeated rows. A zero count here does not prove that every value is valid, so we'll check a few important columns too.

In [ ]:
print("Missing values by column:")
display(df.isna().sum())

print("Exact duplicate rows across every column:", df.duplicated().sum())
print("Columns:", df.columns.tolist())

In [ ]:
df.head(3).T

In [ ]:
df["Unnamed: 0"].equals(df["Id"])

The columns Unnamed: 0 and Id match in this dataset. We'll ignore both for the next repeated-row check.

In [ ]:
print(
    "Rows matching on all non-ID columns:",
    df.drop(columns=["Unnamed: 0", "Id"]).duplicated().sum()
)

## 3. Check the main numerical columns

Look at the range of year, price, distance travelled, and car age. A very high or low value is a reason to check the record, not proof that it is wrong.

In [ ]:
df[["year", "price", "distance_travelled(kms)", "car_age"]].agg(
    ["min", "median", "max"]
)

In [ ]:
df.nlargest(5, "distance_travelled(kms)")[[
    "brand", "full_model_name", "year", "price",
    "distance_travelled(kms)", "city"
]]

In [ ]:
df["distance_travelled(kms)"].quantile([0.95, 0.99, 0.995])

In [ ]:
high_mileage_cars = df.loc[
    df["distance_travelled(kms)"] > 200000,
    ["brand", "full_model_name", "year", "price",
     "distance_travelled(kms)", "city"]
]
print("Cars above 200,000 km:", len(high_mileage_cars))
display(high_mileage_cars)

In [ ]:
high_mileage_cars[
    high_mileage_cars.duplicated(
        subset=["brand", "full_model_name", "year", "price",
                "distance_travelled(kms)", "city"],
        keep=False
    )
]

In the original check, 9 cars had mileage above 200,000 km. Two pairs within that group had matching car details. Matching details may indicate repeated listings, but alone they cannot confirm that the rows are the same physical car.

## 4. Prepare the analysis DataFrame

Keep columns that help answer our main question: how do used-car prices differ across common car details? The dataset also has precomputed scoring columns, but their formulas are not explained here, so they are left out.

There are 425 rows that match on the selected car details. For the main charts, this notebook keeps one copy of each matching set. This is a working choice, not proof that all matching rows are duplicate listings.

In [ ]:
columns = [
    "year", "brand", "full_model_name", "model_name", "price",
    "distance_travelled(kms)", "fuel_type", "city", "car_age"
]

car_eda = df[columns].copy()
car_eda = car_eda.rename(columns={"distance_travelled(kms)": "distance_km"})

numeric_cols = ["year", "price", "distance_km", "car_age"]
for col in numeric_cols:
    car_eda[col] = pd.to_numeric(car_eda[col], errors="coerce")

matching_rows = int(car_eda.duplicated().sum())
car_eda = car_eda.drop_duplicates().reset_index(drop=True)
car_eda["price_lakh"] = car_eda["price"] / 100000

print("Rows in original dataset:", len(df))
print("Rows matching on selected car details:", matching_rows)
print("Rows used for charts:", len(car_eda))

In [ ]:
print("Missing values after preparation:")
display(car_eda.isna().sum())

print("Exact repeated rows in analysis data:", car_eda.duplicated().sum())
print("Price <= 0:", (car_eda["price"] <= 0).sum())
print("Negative mileage:", (car_eda["distance_km"] < 0).sum())
print("Year outside 1900–2026:", (~car_eda["year"].between(1900, 2026)).sum())
print("Negative car age:", (car_eda["car_age"] < 0).sum())

display(car_eda[["year", "price", "distance_km", "car_age"]].describe().round(2))

## 5. Explore prices and relationships

### Chart 1 — Price distribution

The chart hides only the top 1% of prices to make the main group easier to see. The analysis DataFrame is not filtered by this chart setting.

In [ ]:
price_limit = car_eda["price_lakh"].quantile(0.99)
price_plot = car_eda[car_eda["price_lakh"] <= price_limit]

sns.histplot(data=price_plot, x="price_lakh", bins=30)
plt.title("Distribution of Used Car Prices (up to 99th percentile)")
plt.xlabel("Price (₹ lakh)")
plt.ylabel("Number of cars")
plt.tight_layout()
plt.show()

### Chart 2 — Median price by fuel type

The median helps reduce the effect of a few very expensive cars. Check the number of listings too, especially for fuel types with very few records.

In [ ]:
fuel_summary = (
    car_eda.groupby("fuel_type", as_index=False)
    .agg(median_price_lakh=("price_lakh", "median"),
         number_of_cars=("price_lakh", "size"))
    .sort_values("median_price_lakh", ascending=False)
)

sns.barplot(data=fuel_summary, x="median_price_lakh", y="fuel_type")
plt.title("Median Used Car Price by Fuel Type")
plt.xlabel("Median price (₹ lakh)")
plt.ylabel("Fuel type")
plt.tight_layout()
plt.show()
display(fuel_summary.round(2))

### Chart 3 — Median price for the 10 most-listed brands

In [ ]:
top_brands = car_eda["brand"].value_counts().head(10).index

brand_summary = (
    car_eda[car_eda["brand"].isin(top_brands)]
    .groupby("brand", as_index=False)
    .agg(median_price_lakh=("price_lakh", "median"),
         number_of_cars=("price_lakh", "size"))
    .sort_values("median_price_lakh", ascending=True)
)

sns.barplot(data=brand_summary, x="median_price_lakh", y="brand")
plt.title("Median Price for the 10 Most-Listed Brands")
plt.xlabel("Median price (₹ lakh)")
plt.ylabel("Brand")
plt.tight_layout()
plt.show()
display(brand_summary.round(2))

### Chart 4 — Price versus distance travelled

This scatter plot focuses on the central 99% range of both values, making the main pattern easier to see. The excluded points stay in the analysis data.

In [ ]:
distance_limit = car_eda["distance_km"].quantile(0.99)
price_limit = car_eda["price_lakh"].quantile(0.99)
plot_data = car_eda[
    (car_eda["distance_km"] <= distance_limit) &
    (car_eda["price_lakh"] <= price_limit)
]

sns.scatterplot(
    data=plot_data, x="distance_km", y="price_lakh",
    hue="fuel_type", alpha=0.7
)
plt.title("Used Car Price vs Distance Travelled (up to 99th percentiles)")
plt.xlabel("Distance travelled (km)")
plt.ylabel("Price (₹ lakh)")
plt.tight_layout()
plt.show()

### Chart 5 — Median price by manufacturing year

In [ ]:
year_summary = (
    car_eda.groupby("year", as_index=False)
    .agg(median_price_lakh=("price_lakh", "median"),
         number_of_cars=("price_lakh", "size"))
    .sort_values("year")
)

sns.lineplot(data=year_summary, x="year", y="median_price_lakh", marker="o")
plt.title("Median Used Car Price by Manufacturing Year")
plt.xlabel("Manufacturing year")
plt.ylabel("Median price (₹ lakh)")
plt.tight_layout()
plt.show()
display(year_summary.tail(5).round(2))

### Chart 6 — Median price across the 10 most-listed cities

In [ ]:
top_cities = car_eda["city"].value_counts().head(10).index

city_summary = (
    car_eda[car_eda["city"].isin(top_cities)]
    .groupby("city", as_index=False)
    .agg(median_price_lakh=("price_lakh", "median"),
         number_of_cars=("price_lakh", "size"))
    .sort_values("median_price_lakh", ascending=True)
)

sns.barplot(data=city_summary, x="median_price_lakh", y="city")
plt.title("Median Used Car Price in the 10 Most-Listed Cities")
plt.xlabel("Median price (₹ lakh)")
plt.ylabel("City")
plt.tight_layout()
plt.show()
display(city_summary.round(2))

## 6. Key findings

Run this cell after the chart cells. It prints key values from the dataset that was loaded, so the findings reflect the data used when the notebook is run.

In [ ]:
print("USED CAR EDA — KEY FINDINGS")
print("-" * 34)
print(f"Original rows: {len(df):,}")
print(f"Rows used for charts after keeping one copy of matching car details: {len(car_eda):,}")
print(f"Rows matching on selected car details: {matching_rows:,}")
print(f"Median price in analysis data: ₹{car_eda['price'].median():,.0f}")
print(f"Price range: ₹{car_eda['price'].min():,.0f} to ₹{car_eda['price'].max():,.0f}")
print(f"Median distance travelled: {car_eda['distance_km'].median():,.0f} km")
print(f"Distance range: {car_eda['distance_km'].min():,.0f} to {car_eda['distance_km'].max():,.0f} km")
print(f"Manufacturing years: {int(car_eda['year'].min())} to {int(car_eda['year'].max())}")

print("\nMost-listed brand:")
display(car_eda["brand"].value_counts().head(1))

print("Most common fuel type:")
display(car_eda["fuel_type"].value_counts().head(1))

print("City with most listings:")
display(car_eda["city"].value_counts().head(1))

print("Fuel type with highest median price:")
display(fuel_summary.loc[[fuel_summary["median_price_lakh"].idxmax()]])

print("Most-listed brand with highest median price (among top 10 brands):")
display(brand_summary.loc[[brand_summary["median_price_lakh"].idxmax()]])

print("City with highest median price (among top 10 cities):")
display(city_summary.loc[[city_summary["median_price_lakh"].idxmax()]])

## 7. Summary and limitations

- The original dataset has 1,725 rows and 23 columns. No missing values were found in the supplied data.
- Unnamed: 0 and Id contain the same sequence. When ID columns are ignored, 425 rows match on the remaining fields. One copy of each matching set is used for the charts; the source does not confirm whether each match is the same physical listing.
- The highest recorded mileage is 790,000 km. High mileage alone does not prove a record is incorrect.
- Median prices vary across brands, fuel types, years, and cities, but these differences do not prove that one factor causes the price difference.
- Some categories have very few listings. Check the listing count before drawing strong conclusions.
- These results describe listings in this dataset, not the entire used-car market.

## Conclusion

EDA helped me check the data, review unusually high mileage, identify rows with matching car details, and compare used-car prices across important features. The next step is to use a suitable dataset to learn and build a predictive Machine Learning model.